# Lemmas + Embeddings


*   Embeddings -> Semántica
*   Lemmas -> Gramática


El POC del proyecto, contiene los mòdulos necesarios para preprocesamiento, extracción de features y una primera clasificación con embeddings a partir del “plot” de la base, combinando comprensión semántica profunda (embeddings) y señales léxicas (TF‑IDF sobre lemmas) mediante un clasificador multilabel.
El pipeline diseñado, es híbrido, contando con dos ramas paralelas (embeddings semánticos y TF‑IDF sobre texto lematizado) que se fusionan y alimentan a un clasificador multilabel para predecir géneros audiovisuales.

## Estudiantes:
- Luis Jorge García
- Luis Eduardo Uribe
- Felipe Barreto Patiño


In [1]:
# !pip install pandas numpy scikit-learn sentence-transformers spacy openpyxl scipy
# !python -m spacy download en_core_web_sm
# !python -m pip install beautifulsoup4

In [1]:
# Este modelo está construido como MVP del Pipeline para categorización de contenido audiovisual a partir
# de información textual del mismojupyter kernelspec list (sinopsis, plot).

# Importing
import re
import numpy as np
import pandas as pd
import unicodedata
import spacy

from spacy.lang.en.stop_words import STOP_WORDS
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report
from scipy.sparse import hstack





c:\repos\nlp-dgo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Loading Data:
df = pd.read_excel("KLUSTERS.xlsx", engine="openpyxl")

df = df[["plot", "genres"]].dropna().reset_index(drop=True)


# El uso de SpaCy preentrenado (inglés) nos permite tener NER y otras herramientas que simplifican desde el preprocesamiento, hasta
# el análisis gramatical y semántico.

def load_spacy_english():
    return spacy.load(
        "en_core_web_sm",
        enable=[
            "tok2vec",
            "tagger",
            "attribute_ruler",
            "lemmatizer",
            "ner",
            "parser"
        ]
    )
# La idea es cargar SpaCy para tokenización, lematización, PoS tagging y uso de NER
NLP_MODEL = load_spacy_english()

# Entidades que sí conviene normalizar
# Sin GPE,LOC,ORG por señales que podemos tener... Como decir "CIA"
NER_REPLA = {
    "PERSON": "_PERSON_",
    "DATE": "_DATE_",
    "TIME": "_TIME_",
    "CARDINAL": "_NUMBER_",
    "ORDINAL": "_ORDINAL_"
}

# Stopwords a conservar
KEEP_STOPWORDS = {"no", "not", "never", "without", "against"}

# PoS útiles para género, Limpiamos ruido, que estos PoS son los que más enriquecen el modelo.
ALLOWED_POS = {"NOUN", "PROPN", "VERB", "ADJ"}

def normalize_text(text: str) -> str:
    text = str(text)

    # quitar metadata tipo ::autor
    text = re.sub(r"::.*$", "", text).strip()

    # normalización unicode
    text = unicodedata.normalize("NFKC", text)

    # espacios repetidos
    text = re.sub(r"\s+", " ", text).strip()

    return text


def preprocess_text(text: str) -> str: # Devolver un string de lemmas limpios.
    text = normalize_text(text)

    # Sin lower() antes de spaCy, detectanos nombres, de una mejor manera.
    doc = NLP_MODEL(text)

    lemmas = [] # inicialización de la lista de lemmas
    i = 0

    while i < len(doc): # recorremos los token del plot
        token = doc[i]

        # ruido
        if token.is_space or token.like_url or token.like_email:
            i += 1
            continue

        # manejar entidades multi-token
        if token.ent_iob_ == "B":
            ent_type = token.ent_type_

            # normalizar solo ciertas entidades
            if ent_type in NER_REPLA:
                lemmas.append(NER_REPLA[ent_type]) # generalizamos las entidades, John-> con el placeholder _PERSON_

                i += 1
                while i < len(doc) and doc[i].ent_iob_ == "I": # avanzamos mientras sigamos ne la misma entidad, como "York", de "New York"
                    i += 1
                continue

        # saltar restos internos de entidad si quedaron
        if token.ent_iob_ == "I":
            i += 1
            continue

        # quitar puntuación ya que no aporta a los lemmas o al TF‑IDF
        if token.is_punct:
            i += 1
            continue

        # conservar algunas negaciones en las stopwords
        if token.is_stop and token.lower_ not in KEEP_STOPWORDS:
            i += 1
            continue

        # filtrar por PoS útiles
        if token.pos_ not in ALLOWED_POS:
            i += 1
            continue

        lemma = token.lemma_.strip().lower() # lemma sin espacios, en minúscula

        # spaCy a veces devuelve -PRON-
        if lemma == "-pron-":
            lemma = token.lower_

        # conservar placeholders NER

        if lemma.upper() in NER_REPLA.values():
            lemmas.append(lemma.upper())
            i += 1
            continue


        # conservar palabras alphab. y compuestos con guion, ya que hay unas como "sci-fi"
        if re.fullmatch(r"[a-zA-Z]+(?:-[a-zA-Z]+)*", lemma):
            lemmas.append(lemma)

        i += 1

    return " ".join(lemmas)



In [3]:
# Aplicando la función de preprocesamiento a la columna "plot_lemmas"
df["plot_lemmas"] = df["plot"].astype(str).apply(preprocess_text)

In [4]:
# Lista de géneros
def parse_genres(x):
    if isinstance(x, str):
        return eval(x)
    return x

df["genres"] = df["genres"].apply(parse_genres)


#  convierte cada fila en un vector binario,
# por cada género (si es scifi, entonces 1 0 0 0 0 0... y así)
mlb = MultiLabelBinarizer()
y = mlb.fit_transform(df["genres"])

In [5]:
# TRAIN / TEST (Gramática)
# Separación de datos train (70%) / test (30%)
#X_text_train, X_text_test, y_train, y_test = train_test_split(
#    df["plot"],
#    y,
#    test_size=0.3,
#    random_state=42
#)
#
#X_lemmas_train = df.loc[X_text_train.index, "plot_lemmas"]
#X_lemmas_test  = df.loc[X_text_test.index,  "plot_lemmas"]

In [6]:
from sklearn.model_selection import train_test_split

# 70% train, 30% temp
X_text_train, X_text_temp, y_train, y_temp = train_test_split(
    df["plot"],
    y,
    test_size=0.30,
    random_state=42
)

# 15% val, 15% test (del 30% total)
X_text_val, X_text_test, y_val, y_test = train_test_split(
    X_text_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

X_lemmas_train = df.loc[X_text_train.index, "plot_lemmas"]
X_lemmas_val   = df.loc[X_text_val.index,   "plot_lemmas"]
X_lemmas_test  = df.loc[X_text_test.index,  "plot_lemmas"]


In [8]:
# Cargamos all-MiniLM-L6-v2 de la librería "SentenceTransformers" (Hugging Face),
# que nos permite convertir texto en embeddings que capturan significado semántico,
# trayendo similitudes semánticas entre frases.

emb_model = SentenceTransformer("all-MiniLM-L6-v2") # all-mpnet-base-v2?

# Train
X_train_emb = emb_model.encode(
    X_text_train.tolist(),
    show_progress_bar=True
)
# Test
X_test_emb = emb_model.encode(
    X_text_test.tolist(),
    show_progress_bar=True
)

# val
X_val_emb = emb_model.encode(
    X_text_val.tolist(),
    show_progress_bar=True)


c:\repos\nlp-dgo\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lucho\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 48/48 [00:22<00:00,  2.09it/s]


In [9]:
# Capturamos palabras clave, patrones en el corpus, con TF-IDF, que nos permite ver
# la frecuencia de palabras y que tan raras son.


tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 3),   # unigramas + bigramas + trigramas
    min_df=3,
    max_df=0.90,
    sublinear_tf=True
)


X_train_tfidf = tfidf.fit_transform(X_lemmas_train)
X_val_tfidf   = tfidf.transform(X_lemmas_val)
X_test_tfidf  = tfidf.transform(X_lemmas_test)




In [10]:
# Normalización de los embeddings para que tengan la misma escala
# que TF-IDF.


from sklearn.preprocessing import StandardScaler, MaxAbsScaler
from scipy.sparse import hstack

# Escalar embeddings
scaler_emb = StandardScaler(with_mean=False)
X_train_emb = scaler_emb.fit_transform(X_train_emb)
X_val_emb   = scaler_emb.transform(X_val_emb)
X_test_emb  = scaler_emb.transform(X_test_emb)


In [11]:
# FINAL
#[semántica, léxico / gramática]

# Fusionar
X_train_final = hstack([X_train_emb, X_train_tfidf])
X_val_final   = hstack([X_val_emb,   X_val_tfidf])
X_test_final  = hstack([X_test_emb,  X_test_tfidf])

# Escalar la matriz final
scaler_final = MaxAbsScaler()
X_train_final = scaler_final.fit_transform(X_train_final)
X_val_final   = scaler_final.transform(X_val_final)
X_test_final  = scaler_final.transform(X_test_final)


In [12]:
# Entreno de un modelo de LogisticRegression por cada género para
#  resolver el problema multietiqueta.

clf = OneVsRestClassifier(
    LogisticRegression(
        max_iter=2000,
        n_jobs=-1,
        class_weight="balanced",
        C=2.0,
        solver="liblinear"
    )
)


# Ajusta los modelos por género, usando los datos de entrenamiento (60% de la base) para aprender a predecir los géneros.
clf.fit(X_train_final, y_train)







c:\repos\nlp-dgo\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\repos\nlp-dgo\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\repos\nlp-dgo\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
c:\repos\nlp-dgo\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.wa

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...r='liblinear')
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",2.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=

In [13]:

#y_proba = clf.predict_proba(X_test_final)
#threshold = 0.35
#y_pred = (y_proba >= threshold).astype(int)


In [14]:
from sklearn.metrics import f1_score

# Buscamos encontrar el mejor threshold para cada género usando el set de validación.

# Probabilidades en validación
y_val_proba = clf.predict_proba(X_val_final) # Prob. ordenadas [0.72, 0.10, 0.41, ...],   por peli

thresholds = np.arange(0.10, 0.80, 0.05) # threshold desde 0.1 a 0.8 ? de paso de 0.05
best_thresholds = []

for i in range(y_val.shape[1]): # recorremos cada género
    best_thr = 0.5 # valor inicial del best threshold. Punto medio.
    best_f1 = 0.0 # valor inicial del best f1 score.

    for thr in thresholds: # Probamos todos los thresholds para este género
        pred_i = (y_val_proba[:, i] >= thr).astype(int)
        f1 = f1_score(y_val[:, i], pred_i, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr

    best_thresholds.append(best_thr) # se guarda el mejor threshold

best_thresholds = np.array(best_thresholds) # a array

In [15]:
# Cálculo las prob de cada género para cada ejemplo del set de test.
y_test_proba = clf.predict_proba(X_test_final)
#y_pred = (y_test_proba >= best_thresholds).astype(int) noo

classes = np.array(mlb.classes_) # nombres de géneros, en array.

# thresholds aprendidos en validación. dic por género
threshold_map = dict(zip(classes, best_thresholds))

# la idea es modificar manualmente el threshold de los géneros que pueden generar ruido.
for g in ["Sci-Fi", "Short", "Horror", "Fantasy"]:
    if g in threshold_map:
        threshold_map[g] = max(threshold_map[g], 0.50) # like: "Horror": 0.25  ->  0.50


# favorecer drama
if "Drama" in threshold_map:
    threshold_map["Drama"] = min(threshold_map["Drama"], 0.30)

threshold_vec = np.array([threshold_map[g] for g in classes])


In [16]:
# ¿Qué géneros finales dejar por cada película?

def decode_row(probs, threshold_vec, min_ratio=0.80, max_labels=3): # Procesar las prob. max 3 gens por peli
    # candidatos por threshold
    active = np.where(probs >= threshold_vec)[0].tolist() # se toma la posición de donde la comparación entre cada prob >= threshold

    # top-1 siempre
    top1 = int(np.argmax(probs)) # Busca el índice del género con mayor prob.
    top1_prob = probs[top1] # Guarda la prob. del género principal.

    if not active:
        return [top1] # Si ningún género pasó su threshold, entonces devuelve solo el género más probable.

    if top1 not in active:
        active = [top1] + active # Si el género principal no estaba en la lista de activos, lo agrega al principio.

    # solo mantener géneros suficientemente cercanos al principal
    filtered = [
        idx for idx in active
        if idx == top1 or probs[idx] >= top1_prob * min_ratio # siempre el top1 y además los géneros cuya prob. esté lo bastante cerca de la del principal.
        #80% de la prob del principal
    ]

    # máximo de labels
    filtered = sorted(filtered, key=lambda i: probs[i], reverse=True)[:max_labels] # Ordena los géneros filtrados de mayor a menor prob.

    return filtered
##############################################################################

# Probando en test:

pred_rows = []
for i in range(len(y_test_proba)):
    idxs = decode_row(y_test_proba[i], threshold_vec, min_ratio=0.80, max_labels=3)
    row = np.zeros(len(classes), dtype=int)
    row[idxs] = 1
    pred_rows.append(row)

y_pred = np.vstack(pred_rows) # Predicción final

In [17]:
# Métricas

#y_pred = clf.predict(X_test_final)

print(
    classification_report(
        y_test,
        y_pred,
        target_names=mlb.classes_
    )
)

              precision    recall  f1-score   support

      Action       0.75      0.54      0.63       301
   Adventure       0.73      0.56      0.63       250
   Animation       0.78      0.56      0.65       139
   Biography       0.66      0.48      0.56        87
      Comedy       0.78      0.59      0.67       528
       Crime       0.78      0.55      0.64       236
 Documentary       0.81      0.60      0.69       108
       Drama       0.78      0.58      0.67       685
      Family       0.79      0.66      0.72       208
     Fantasy       0.73      0.43      0.54       168
   Film-Noir       0.00      0.00      0.00         4
   Game-Show       0.67      0.29      0.40         7
     History       0.69      0.44      0.54        66
      Horror       0.71      0.70      0.71       113
       Music       0.66      0.70      0.68        53
     Musical       0.42      0.19      0.26        42
     Mystery       0.69      0.47      0.56       163
        News       0.89    

c:\repos\nlp-dgo\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [18]:
# Exportar Resultados en .CSV

true_genres = mlb.inverse_transform(y_test)
pred_genres = mlb.inverse_transform(y_pred)

results_df = pd.DataFrame({
    "plot": X_text_test.values,
    "true_genres": [", ".join(g) for g in true_genres],
    "predicted_genres": [", ".join(g) for g in pred_genres]
})

results_df.to_csv(
    "Last_resultados_modelo_hibrido_embeddings_tfidf.csv",
    index=False
)

In [19]:
thresholds_df = pd.DataFrame({
    "genre": mlb.classes_,
    "best_threshold": best_thresholds
})

thresholds_df.to_csv("thresholds_por_genero.csv", index=False)